# Phase 9: Reproducibility & Experimental Specification
**SmartGrid Sentinel Journal Experiment Series - Notebook 08**

## 1. Reproducibility Objective
Full scientific reproducibility is essential for peer-reviewed academic publication.
This notebook documents every technical configuration parameter, data preprocessing protocol, random seed constraint, model hyperparameter, training regime, and hardware/software environment required to replicate all empirical results reported in the SmartGrid Sentinel study.

## 2. Environment Dependencies & Setup

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

rep_dir = '../reproducibility_phase9'
if not os.path.exists(rep_dir):
    rep_dir = 'reproducibility_phase9'

print('Environment initialized.')

Environment initialized.


## 3. Master Reproducibility Specification Summary

Loading the experimental configuration metadata recorded in `reproducibility_config.json` and `reproducibility_summary.csv`:

In [2]:
with open(os.path.join(rep_dir, 'reproducibility_config.json'), 'r', encoding='utf-8') as f:
    rep_config = json.load(f)

rep_csv = pd.read_csv(os.path.join(rep_dir, 'reproducibility_summary.csv'))
print('=== MASTER REPRODUCIBILITY SUMMARY TABLE ===')
print(rep_csv.head(15).to_string(index=False))

=== MASTER REPRODUCIBILITY SUMMARY TABLE ===
                      Category                      Parameter                                                     Value
            1. Dataset Version                   Dataset File                               datasetNew/mergeDataset.csv
            1. Dataset Version                    Raw Records                                                     65983
            1. Dataset Version Shifted Valid Sequence Records                                                     64983
            1. Dataset Version                 Feeder Streams                                                       143
            1. Dataset Version                      Districts                                                        15
            1. Dataset Version                 Missing Values                                                         0
            1. Dataset Version                 Duplicate Rows                                                      

## 4. Technical Experimental Specifications

### A. Dataset & Version Source
- **Dataset Path:** `datasetNew/mergeDataset.csv`.
- **Raw Telemetry Logs:** 65,983 records across 143 Upazila feeder streams and 15 districts.
- **Target Shifted Valid Sequences:** 64,983 records post 1-step target shifting ($T + 2\text{h}$ target risk level).
- **Data Quality:** 0 missing values, 0 duplicate rows.

### B. Feature Vector Configuration (28 Total Features)
- **Core Physical & Telemetry (15):** `hour`, `weekday`, `temperature`, `humidity`, `rainfall`, `wind_speed`, `weather_state`, `electricity_demand`, `renewable_generation`, `transformer_load`, `district`, `upazila`, `area_type`, `substation_id`, `feeder_id`.
- **Grid Asset Attributes (6):** `transformer_age`, `transformer_capacity`, `outage_history`, `maintenance_due`, `population_density`, `industrial_load_ratio`.
- **Engineered Domain Indicators (7):** `load_utilization`, `demand_utilization`, `renewable_ratio`, `thi`, `wind_temp_interaction`, `is_peak_hour`, `is_weekend`.

### C. Train/Validation/Test Split & Scaling Protocol
- **Chronological Split Strategy:** Data split chronologically per substation/feeder stream to prevent temporal data leakage.
- **Raw Data Split:** 80% Train (`train_df`), 20% Test (`test_df`).
- **Sequence Windows:** 42,674 Train windows (80% of `train_df`), 10,545 Validation windows (20% of `train_df`), 10,027 Test windows (`test_df`).
- **Effective Partition Share:** **67% Train / 17% Validation / 16% Test**.
- **Scaling Procedure:** `StandardScaler` fitted strictly on the 80% chronological training partition and applied to validation/test sequence tensors.

## 5. Hyperparameters, Seeds & Compute Environment

### D. Deterministic Random Seeds
- `SEED = 42` enforced across Python `random`, NumPy `np.random`, PyTorch `torch.manual_seed`, PyTorch CUDA `torch.cuda.manual_seed_all`, TensorFlow `tf.random.set_seed`, `cudnn.deterministic=True`.

### E. Sequence Model Architecture (Informer)
- **Input Channels ($c_{\text{in}}$):** 28, **Output Classes ($c_{\text{out}}$):** 3 (`Low`, `Medium`, `High`).
- **Observation Window ($seq\_len$):** 5 steps (10-hour historical lookback at 2-hour sampling resolution).
- **Embedding Dimension:** 64, **Multi-Head Self-Attention:** 4 heads (`batch_first=True`).
- **Fully Connected Layers:** Dense ($64 \times 5 = 320 \to 128 \to 3$), Dropout = 0.2, ReLU activation.

### F. Training & Optimization Settings
- **Batch Sizes:** Train = 64, Validation = 512, Test = 64.
- **Optimizer:** `optim.Adam` (Learning Rate $lr = 0.001$, $\beta_1=0.9, \beta_2=0.999$, $\epsilon=10^{-8}$).
- **Loss Function:** `nn.CrossEntropyLoss()`.
- **Epochs & Checkpoints:** 50 Total Epochs; checkpoints saved every 10 epochs (`epoch_10.pth` selected for operational API deployment).
- **Early Stopping:** Monitored `val_loss` with patience = 10 epochs.

### G. Software Dependencies & Hardware Environment
- **OS & Compute Platform:** Windows 11, AMD64 Family 25 Model 80 (6 Physical / 12 Logical Cores), 7.34 GB System RAM.
- **Core Libraries:** Python 3.13.12, PyTorch 2.13.0+cpu, Scikit-Learn 1.7.2, XGBoost 3.2.0, Pandas 3.0.3, NumPy 2.4.6, Joblib 1.5.3, FastAPI 0.115.0.

In [3]:
# Print Key Reproducibility Checklist Summary
print('=== REPRODUCIBILITY CHECKLIST SUMMARY ===')
for category, content in rep_config.items():
    print(f'\n--- {category} ---')
    if isinstance(content, dict):
        for k, v in content.items():
            print(f'  {k}: {v}')
    elif isinstance(content, list):
        print(f'  Features ({len(content)}): {content[:5]}...')

=== REPRODUCIBILITY CHECKLIST SUMMARY ===

--- 1. Dataset Version ---
  Dataset File: datasetNew/mergeDataset.csv
  Raw Records: 65983
  Shifted Valid Sequence Records: 64983
  Feeder Streams: 143
  Districts: 15
  Missing Values: 0
  Duplicate Rows: 0

--- 2. Train/Validation/Test Split ---
  Split Strategy: Chronological Sequence Split per Substation/Feeder Stream
  Raw Data Split Ratio: 80% Train (train_df), 20% Test (test_df)
  Train Sequence Windows: 42674
  Validation Sequence Windows: 10545
  Test Sequence Windows: 10027
  Effective Percentage: 67% Train / 17% Validation / 16% Test

--- 3. Random Seeds ---
  Global Seed: 42
  Python random.seed: 42
  NumPy np.random.seed: 42
  PyTorch torch.manual_seed: 42
  PyTorch CUDA torch.cuda.manual_seed_all: 42
  TensorFlow tf.random.set_seed: 42
  PyTorch Deterministic Flags: cudnn.deterministic=True, cudnn.benchmark=False

--- 4. Feature List (28 Total) ---
  Features (28): ['hour', 'weekday', 'temperature', 'humidity', 'rainfall']...



## 6. Journal-Ready Reproducibility Statement

> **Experimental Replication Protocol:** All experiments reported in this paper are fully reproducible using the configuration parameters, random seeds (`SEED = 42`), and dataset partitions documented in `reproducibility_config.json`. The codebase, preprocessed datasets (`datasetNew/mergeDataset.csv`), trained model weights (`epoch_10.pth`), scaler artifacts (`scaler.pkl`), and evaluation scripts have been archived and version-controlled under git tag `v1_conference` to guarantee complete scientific transparency.